In [1]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("github-token")

github_token = secret_value_0

!git clone https://{github_token}@github.com/manaal-m/acdc-cardiac-mri-segmentation.git
%cd acdc-cardiac-mri-segmentation

Cloning into 'acdc-cardiac-mri-segmentation'...
remote: Enumerating objects: 29, done.
remote: Counting objects: 100% (29/29), done.
remote: Compressing objects: 100% (26/26), done.
remote: Total 29 (delta 1), reused 25 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (29/29), 289.71 KiB | 10.73 MiB/s, done.
Resolving deltas: 100% (1/1), done.
/kaggle/working/acdc-cardiac-mri-segmentation


In [ ]:
%pip install -q lightning nibabel segmentation-models-pytorch kagglehub torchinfo

import os
import json
import torch
import pytorch_lightning as pl
from pytorch_lightning import Trainer
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping

from src.dataset import ACDC2DDataModule
from src.model import LitUNet2D
from src.benchmark import measure_batch_latency, model_size_mb, load_results, save_results, update_result

In [ ]:
print("Imports succeeded.")

In [ ]:
import kagglehub

data_path = kagglehub.dataset_download("samdazel/automated-cardiac-diagnosis-challenge-miccai17")
acdc_root = f"{data_path}/database/training"

print("Data downloaded to:", data_path)

In [ ]:
# Config
ENCODERS = ["resnet18", "vgg16", "mobilenet_v2", "resnet50"]
MAX_EPOCHS = 50
BATCH_SIZE = 8
LR = 1e-3
SEEDS = [42, 0, 11]
all_seed_results = {}

CHECKPOINT_DIR = "checkpoints"
RESULTS_PATH = "results/all_results.json"

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs("results", exist_ok=True)

In [ ]:
import numpy as np

for seed in SEEDS:
    pl.seed_everything(seed, workers=True)
    results = load_results(RESULTS_PATH)

    dm = ACDC2DDataModule(acdc_root, batch_size=BATCH_SIZE, num_workers=2, seed=seed)
    dm.setup()

    print(f"\n{'='*40}")
    print(f"SEED {seed}")
    print(f"{'='*40}")
    seed_results = {}

    for encoder in ENCODERS:
        print(f"\nTraining: {encoder}")

        model = LitUNet2D(lr=LR, encoder_name=encoder)

        early_stop = EarlyStopping(monitor="val_dice", mode="max", patience=20, verbose=True)
        checkpoint = ModelCheckpoint(
            monitor="val_dice", mode="max", save_top_k=1,
            dirpath=CHECKPOINT_DIR,
            filename=f"seed{seed}_{encoder}-acdc-{{epoch:02d}}-{{val_dice:.4f}}",
        )
        trainer = Trainer(
            max_epochs=MAX_EPOCHS, callbacks=[early_stop, checkpoint],
            log_every_n_steps=5, enable_progress_bar=True, logger=False,
        )

        trainer.fit(model, dm)

        best_model = LitUNet2D.load_from_checkpoint(checkpoint.best_model_path)
        best_model = best_model.to("cuda")

        final_ckpt_path = os.path.join(CHECKPOINT_DIR, f"seed{seed}_{encoder}_best.pth")
        torch.save(best_model.state_dict(), final_ckpt_path)

        best_model.eval()
        batch_latency_ms = measure_batch_latency(best_model, dm.val_dataloader(), device="cuda", n_batches=50)

        test_res = trainer.test(best_model, datamodule=dm, verbose=False)
        size_mb = model_size_mb(best_model)

        seed_results[encoder] = {
            "dice": round(test_res[0].get("test_dice", -1), 4),
            "iou":  round(test_res[0].get("test_miou", -1), 4),
        }

        update_result(
            results, encoder,
            dice=seed_results[encoder]["dice"],
            iou=seed_results[encoder]["iou"],
            inference_ms_batch8_mean=round(batch_latency_ms, 2),
            model_size_mb=size_mb,
        )
        save_results(results, RESULTS_PATH)
        print(json.dumps(results[encoder], indent=2))

    all_seed_results[seed] = seed_results

print("\nALL TRAINING RUNS COMPLETE")

print(f"\n{'Encoder':<18} {'Dice mean':>12} {'Dice std':>10} {'mIoU mean':>12} {'mIoU std':>10}")
print("-" * 64)
for encoder in ENCODERS:
    dices = [all_seed_results[s][encoder]["dice"] for s in SEEDS]
    ious  = [all_seed_results[s][encoder]["iou"]  for s in SEEDS]
    print(f"{encoder:<18} {np.mean(dices):>12.4f} {np.std(dices):>10.4f} {np.mean(ious):>12.4f} {np.std(ious):>10.4f}")

In [ ]:
CLASS_NAMES = {1: "LV", 2: "RV", 3: "Myocardium"}
per_class_all_seeds = {enc: {name: [] for name in CLASS_NAMES.values()} for enc in ENCODERS}

for seed in SEEDS:
    pl.seed_everything(seed, workers=True)
    dm = ACDC2DDataModule(acdc_root, batch_size=1, num_workers=2, seed=seed)
    dm.setup()
    test_loader = dm.test_dataloader()

    for encoder in ENCODERS:
        ckpt_path = os.path.join(CHECKPOINT_DIR, f"seed{seed}_{encoder}_best.pth")
        model = LitUNet2D(lr=LR, encoder_name=encoder)
        model.load_state_dict(torch.load(ckpt_path, map_location="cuda"))
        model = model.to("cuda").eval()

        all_preds, all_targets = [], []
        with torch.no_grad():
            for x, y in test_loader:
                pred = torch.argmax(model(x.to("cuda")), dim=1).squeeze().cpu()
                all_preds.append(pred)
                all_targets.append(y.squeeze())

        preds = torch.stack(all_preds)
        targets = torch.stack(all_targets)

        for cls, name in CLASS_NAMES.items():
            pred_bin = (preds == cls).float()
            tgt_bin  = (targets == cls).float()
            intersection = (pred_bin * tgt_bin).sum()
            d = (2 * intersection) / (pred_bin.sum() + tgt_bin.sum() + 1e-5)
            per_class_all_seeds[encoder][name].append(d.item())

print(f"{'Encoder':<18} {'LV':>16} {'RV':>16} {'Myocardium':>16}")
print("-" * 68)
for encoder in ENCODERS:
    row = f"{encoder:<18}"
    for name in CLASS_NAMES.values():
        vals = per_class_all_seeds[encoder][name]
        row += f"  {np.mean(vals):.4f}±{np.std(vals):.4f}"
    print(row)

In [ ]:
from torchinfo import summary

dummy = torch.zeros(1, 1, 512, 512).to("cuda")

print(f"{'Encoder':<18} {'Params (M)':>12} {'GMACs':>10}")
print("-" * 42)

for encoder in ENCODERS:
    model = LitUNet2D(lr=LR, encoder_name=encoder).to("cuda")
    s = summary(model, input_data=dummy, verbose=0)
    params = s.total_params / 1e6
    gmacs = s.total_mult_adds / 1e9
    print(f"{encoder:<18} {params:>12.2f} {gmacs:>10.2f}")